# LeetCode 474 — Ones and Zeroes bằng Python

Link đề: https://leetcode.com/problems/ones-and-zeroes/ · Độ khó: **Medium** · Chủ đề: Array, String, Dynamic Programming, Knapsack 0-1 (cái túi hai chiều)

Skeleton LeetCode:

```python
class Solution:
    def findMaxForm(self, strs: List[str], m: int, n: int) -> int:
        ...
```

Ghi chú: bài này **không** phải Premium. Đây là bài "cái túi 0-1 với **hai** sức chứa" (một cho số `0`, một cho số `1`), nên là bước chuyển rất đẹp từ knapsack một chiều sang hai chiều.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chọn **tập con lớn nhất** các chuỗi nhị phân sao cho tổng số ký tự `'0'` không quá `m` và tổng số ký tự `'1'` không quá `n`; hiểu vì sao đây là **bài knapsack 0-1** (mỗi chuỗi: lấy hoặc bỏ) với **hai** loại "trọng lượng".
2. Dùng các công cụ Python cần thiết: `str.count`, ghép tuple, tạo bảng 2 chiều/3 chiều bằng list comprehension (và tránh bẫy `[[0]*n]*m`), `functools.lru_cache`, vòng lặp `range(m, a - 1, -1)` đi **ngược**.
3. Mô hình hóa bằng **DP**: `f[i][j][k]` = số chuỗi nhiều nhất khi xét `i` chuỗi đầu, dùng tối đa `j` số `0`, `k` số `1`; hai lựa chọn: **bỏ** chuỗi `i` hoặc **lấy** chuỗi `i`.
4. Cài đặt 4 cách khác nhau: **vét cạn mọi tập con (`2^L`) → đệ quy có nhớ → DP 3 chiều → DP 2 chiều nén bộ nhớ (duyệt ngược)**, và hiểu vì sao phải duyệt ngược.
5. Tránh các lỗi kinh điển: duyệt xuôi (biến thành knapsack "vô hạn", dùng một chuỗi nhiều lần), nhầm `m` là số `1`, tham lam chọn chuỗi ngắn nhất, quên trường hợp "bỏ chuỗi", và tạo bảng bằng `[[0] * n] * m`.

## 2. Đề bài & ví dụ

Cho mảng các **chuỗi nhị phân** `strs` (mỗi chuỗi chỉ gồm `'0'` và `'1'`) và hai số nguyên `m`, `n`.

Trả về **kích thước của tập con lớn nhất** của `strs` sao cho trong tập con đó có **tối đa `m` số `0`** và **tối đa `n` số `1`** (đếm tổng trên tất cả chuỗi được chọn).

Tập `x` là tập con của `y` nếu mọi phần tử của `x` đều nằm trong `y`. (Mỗi chuỗi trong `strs` được chọn **nhiều nhất một lần**.)

**Ví dụ 1:** `strs = ["10","0001","111001","1","0"]`, `m = 5`, `n = 3` → `4`. Tập lớn nhất là `{"10", "0001", "1", "0"}`: số `0` là `1 + 3 + 0 + 1 = 5`, số `1` là `1 + 1 + 1 + 0 = 3`. Chuỗi `"111001"` có bốn số `1` nên vượt `n = 3`.

**Ví dụ 2:** `strs = ["10","0","1"]`, `m = 1`, `n = 1` → `2` (chọn `{"0", "1"}`).

**Ràng buộc:**

- `1 <= strs.length <= 600`
- `1 <= strs[i].length <= 100`
- `strs[i]` chỉ gồm `'0'` và `'1'`
- `1 <= m, n <= 100`

Với `600` chuỗi và `m = n = 100`, bảng DP có `600 * 101 * 101 ≈ 6 × 10^6` ô: chấp nhận được.

In [1]:
import itertools
import random
import time
from functools import lru_cache
from typing import List

examples = [
    (["10", "0001", "111001", "1", "0"], 5, 3, 4),
    (["10", "0", "1"], 1, 1, 2),
]
for strs, m, n, ans in examples:
    print(f"strs = {strs}, m = {m}, n = {n}  -> {ans}")

strs = ['10', '0001', '111001', '1', '0'], m = 5, n = 3  -> 4
strs = ['10', '0', '1'], m = 1, n = 1  -> 2


## 3. Kiến thức nền

### 3.1. `str.count` để đếm số `0` và số `1`

Mỗi chuỗi có một "giá" gồm hai số: `(số 0, số 1)`. `"0001".count("0")` cho `3`, `"0001".count("1")` cho `1`.

In [2]:
s = "111001"
print(s.count("0"), s.count("1"))                # 2 4
cost = [(t.count("0"), t.count("1")) for t in ["10", "0001", "111001", "1", "0"]]
print(cost)                                      # [(1, 1), (3, 1), (2, 4), (0, 1), (1, 0)]

2 4
[(1, 1), (3, 1), (2, 4), (0, 1), (1, 0)]


### 3.2. Knapsack 0-1 (cái túi) một chiều: nhắc lại ý tưởng

Có các món đồ, mỗi món có **trọng lượng** `w` và **giá trị** `v`; túi chứa tối đa `W`. Mỗi món **lấy hoặc bỏ** (0-1). Muốn tổng giá trị lớn nhất.

Bài này giống hệt, chỉ khác: mỗi chuỗi có **hai** trọng lượng `(a, b) = (số 0, số 1)`, túi có **hai** sức chứa `(m, n)`, và **giá trị mỗi món là 1** (ta đếm số chuỗi).

Ví dụ nhỏ một chiều: món `(w=2, v=3)`, `(w=3, v=4)`, `W = 4`: tốt nhất lấy một món giá trị `4`.

In [3]:
def knapsack_1d(items, W):
    dp = [0] * (W + 1)                           # dp[c] = giá trị lớn nhất với sức chứa c
    for w, v in items:
        for c in range(W, w - 1, -1):            # duyệt NGƯỢC để mỗi món chỉ dùng một lần
            dp[c] = max(dp[c], dp[c - w] + v)
    return dp[W]

print(knapsack_1d([(2, 3), (3, 4)], 4))          # 4
print(knapsack_1d([(2, 3), (3, 4)], 5))          # 7 (lấy cả hai)

4
7


### 3.3. Bảng nhiều chiều và bẫy `[[0] * n] * m`

Để tạo bảng `(m+1) × (n+1)` toàn `0`, dùng list comprehension `[[0] * (n + 1) for _ in range(m + 1)]`. **Không** viết `[[0] * (n + 1)] * (m + 1)`: cách đó tạo `m + 1` tham chiếu tới **cùng một hàng**, sửa một ô là đổi cả cột.

In [4]:
sai = [[0] * 3] * 2
sai[0][1] = 9
print("sai :", sai)                              # [[0, 9, 0], [0, 9, 0]]  <- hai hàng cùng một list!

dung = [[0] * 3 for _ in range(2)]
dung[0][1] = 9
print("dung:", dung)                             # [[0, 9, 0], [0, 0, 0]]

sai : [[0, 9, 0], [0, 9, 0]]
dung: [[0, 9, 0], [0, 0, 0]]


### 3.4. `lru_cache` và `range` đi ngược

`@lru_cache(None)` ghi nhớ kết quả của hàm theo tham số (cho đệ quy có nhớ). `range(m, a - 1, -1)` đếm ngược từ `m` xuống `a`: ví dụ `list(range(5, 1, -1))` ra `[5, 4, 3, 2]`; ở đây `a - 1` là điểm dừng (không bao gồm) nên giá trị nhỏ nhất là `a`.

In [5]:
print(list(range(5, 1, -1)))       # [5, 4, 3, 2]

@lru_cache(None)
def fib(k):
    return k if k < 2 else fib(k - 1) + fib(k - 2)

print(fib(30), fib.cache_info().currsize)

[5, 4, 3, 2]
832040 31


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Ví dụ 2:** `strs = ["10","0","1"]`, `m = 1`, `n = 1`. Giá của từng chuỗi `(0s, 1s)`: `"10" → (1,1)`, `"0" → (1,0)`, `"1" → (0,1)`. Túi `(1,1)`.

Thử các tập con:

- `{"10"}`: `(1,1)` ✓ → kích thước 1.
- `{"0","1"}`: `(1,0) + (0,1) = (1,1)` ✓ → kích thước **2**.
- `{"10","0"}`: `(2,1)` ✗ vượt `m`.
- `{"10","0","1"}`: `(2,2)` ✗.

Đáp án `2`. Nhận xét: chuỗi `"10"` "đắt" (tốn cả hai loại) mà chỉ được tính 1 điểm, trong khi `"0"` và `"1"` "rẻ" nên gộp lại được 2 điểm. Vì vậy **tham lam theo giá thấp** có vẻ hợp lý nhưng không chắc đúng (xem mục 13).

**Cách nghĩ DP:** xét lần lượt từng chuỗi, với mỗi chuỗi có hai lựa chọn: **bỏ** hoặc **lấy** (nếu còn đủ chỗ). Ta cần biết "tốt nhất có thể" cho **mọi** túi con `(j, k)`, vì lấy chuỗi này rồi thì phần chỗ còn lại là `(j - a, k - b)`.

In [6]:
# Thử tay bằng máy: liệt kê mọi tập con của ví dụ 2
strs, m, n = ["10", "0", "1"], 1, 1
for mask in range(1 << len(strs)):
    chon = [strs[i] for i in range(len(strs)) if mask >> i & 1]
    z = sum(t.count("0") for t in chon)
    o = sum(t.count("1") for t in chon)
    print(f"{str(chon):22} zeros={z} ones={o}  hop le={z <= m and o <= n}")

[]                     zeros=0 ones=0  hop le=True
['10']                 zeros=1 ones=1  hop le=True
['0']                  zeros=1 ones=0  hop le=True
['10', '0']            zeros=2 ones=1  hop le=False
['1']                  zeros=0 ones=1  hop le=True
['10', '1']            zeros=1 ones=2  hop le=False
['0', '1']             zeros=1 ones=1  hop le=True
['10', '0', '1']       zeros=2 ones=2  hop le=False


## 5. Cách 1 — Brute Force / baseline: thử MỌI tập con

**Ý tưởng:** có `L = len(strs)` chuỗi, nên có `2^L` tập con. Duyệt từng tập con (bằng bitmask `0 .. 2^L - 1`), tính tổng số `0` và số `1`, nếu `<= m` và `<= n` thì cập nhật đáp án bằng kích thước tập.

Cách này **luôn đúng**, dễ viết, nên dùng làm **oracle** cho `L <= 14`. Với `L = 600` thì `2^600` không thể.

In [7]:
def find_max_form_brute(strs, m, n):
    L = len(strs)
    cost = [(t.count("0"), t.count("1")) for t in strs]
    best = 0
    for mask in range(1 << L):
        z = o = cnt = 0
        for i in range(L):
            if mask >> i & 1:
                z += cost[i][0]
                o += cost[i][1]
                cnt += 1
        if z <= m and o <= n and cnt > best:
            best = cnt
    return best

for strs, m, n, ans in examples:
    print(find_max_form_brute(strs, m, n), "(mong doi", ans, ")")

4 (mong doi 4 )
2 (mong doi 2 )


**Độ phức tạp:** Time `O(2^L * L)`, Space `O(L)`.

**Nút thắt:** ta tính lại những thứ giống nhau. Nhiều tập con khác nhau cùng "dùng hết `(j, k)` chỗ" nhưng ta duyệt chúng độc lập. Chỉ cần nhớ "tốt nhất với `j` số 0 và `k` số 1" là đủ, vì **tương lai chỉ phụ thuộc vào phần chỗ còn lại**, không phụ thuộc cách ta đã chọn. Hãy đo để thấy `2^L` nổ nhanh.

In [8]:
for L in (8, 10, 12, 14):
    strs = ["01"] * L
    t0 = time.perf_counter()
    kq = find_max_form_brute(strs, 100, 100)
    print(f"L = {L:2}: 2^L = {2**L:6} tap con, kq = {kq}, mat {time.perf_counter() - t0:.3f}s")

L =  8: 2^L =    256 tap con, kq = 8, mat 0.000s
L = 10: 2^L =   1024 tap con, kq = 10, mat 0.001s
L = 12: 2^L =   4096 tap con, kq = 12, mat 0.003s
L = 14: 2^L =  16384 tap con, kq = 14, mat 0.016s


## 6. Ý tưởng thuật toán / bước đột phá

Định nghĩa trạng thái: **`f[i][j][k]`** = số chuỗi **nhiều nhất** có thể chọn trong `i` chuỗi đầu tiên (`strs[0..i-1]`) mà tổng số `0` `<= j` và tổng số `1` `<= k`.

Chuyển trạng thái cho chuỗi thứ `i` (có `a` số `0`, `b` số `1`):

- **Bỏ** chuỗi: `f[i][j][k] = f[i-1][j][k]`.
- **Lấy** chuỗi (chỉ khi `j >= a` và `k >= b`): `f[i][j][k] = f[i-1][j-a][k-b] + 1`.

Lấy `max` của hai khả năng. Đáp án: `f[L][m][n]`.

**Vì sao đúng (trực giác):** lấy chuỗi `i` thì phần chỗ còn lại cho `i - 1` chuỗi trước là `(j - a, k - b)`, và vì ta đã lưu "tốt nhất" cho mọi cặp chỗ nên chỉ cần tra bảng. Đây chính là tính chất **cấu trúc con tối ưu** của knapsack.

**Bước đột phá thứ hai (nén bộ nhớ):** hàng `i` chỉ phụ thuộc hàng `i - 1`. Nên chỉ giữ **một bảng 2 chiều** `dp[j][k]` và cập nhật **tại chỗ**; muốn `dp[j - a][k - b]` còn là giá trị của "hàng cũ" thì phải duyệt `j`, `k` **từ lớn về nhỏ**.

## 7. Các cách tiếp theo

### Cách 2 — Đệ quy có nhớ (top-down)

**Từng đoạn:**

1. `go(i, j, k)` = số chuỗi nhiều nhất chọn từ `strs[i:]` với tối đa `j` số `0`, `k` số `1`.
2. Hết chuỗi (`i == L`) → `0`.
3. Bỏ chuỗi `i`: `go(i + 1, j, k)`.
4. Lấy chuỗi `i` nếu `j >= a` và `k >= b`: `1 + go(i + 1, j - a, k - b)`.
5. Trả `max`; `@lru_cache` nhớ theo `(i, j, k)`.

Chú ý độ sâu đệ quy bằng `L` (tối đa 600): vượt giới hạn mặc định 1000? Không, `600 < 1000` nên ổn.

In [9]:
def find_max_form_memo(strs, m, n):
    L = len(strs)
    cost = [(t.count("0"), t.count("1")) for t in strs]

    @lru_cache(None)
    def go(i, j, k):
        if i == L:
            return 0
        best = go(i + 1, j, k)                         # bỏ chuỗi i
        a, b = cost[i]
        if j >= a and k >= b:
            best = max(best, 1 + go(i + 1, j - a, k - b))   # lấy chuỗi i
        return best

    return go(0, m, n)

for strs, m, n, ans in examples:
    print(find_max_form_memo(strs, m, n), "(mong doi", ans, ")")

4 (mong doi 4 )
2 (mong doi 2 )


**Code nộp LeetCode (Cách 2):**

```python
from functools import lru_cache

class Solution:
    def findMaxForm(self, strs: List[str], m: int, n: int) -> int:
        L = len(strs)
        cost = [(t.count("0"), t.count("1")) for t in strs]

        @lru_cache(None)
        def go(i, j, k):
            if i == L:
                return 0
            best = go(i + 1, j, k)
            a, b = cost[i]
            if j >= a and k >= b:
                best = max(best, 1 + go(i + 1, j - a, k - b))
            return best

        return go(0, m, n)
```

**Độ phức tạp:** Time `O(L * m * n)` (số trạng thái), Space `O(L * m * n)` cho cache + `O(L)` đệ quy. Dễ viết nhưng Python gọi hàm chậm: với `600 * 101 * 101` trạng thái dễ chậm; chỉ dùng khi cần tư duy.

### Cách 3 — DP 3 chiều bottom-up `f[i][j][k]`

Dịch trực tiếp công thức ở mục 6. Hàng `0` toàn `0` (không có chuỗi nào thì chọn được `0`). Với mỗi `i` từ `1` đến `L`: đếm `a, b` của `strs[i-1]`, rồi lấp đầy `f[i][j][k]` cho mọi `j`, `k`.

In [10]:
def find_max_form_3d(strs, m, n):
    L = len(strs)
    f = [[[0] * (n + 1) for _ in range(m + 1)] for _ in range(L + 1)]
    for i, s in enumerate(strs, 1):
        a, b = s.count("0"), s.count("1")
        for j in range(m + 1):
            for k in range(n + 1):
                f[i][j][k] = f[i - 1][j][k]                     # bỏ
                if j >= a and k >= b:
                    f[i][j][k] = max(f[i][j][k], f[i - 1][j - a][k - b] + 1)   # lấy
    return f[L][m][n]

for strs, m, n, ans in examples:
    print(find_max_form_3d(strs, m, n), "(mong doi", ans, ")")

4 (mong doi 4 )
2 (mong doi 2 )


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def findMaxForm(self, strs: List[str], m: int, n: int) -> int:
        L = len(strs)
        f = [[[0] * (n + 1) for _ in range(m + 1)] for _ in range(L + 1)]
        for i, s in enumerate(strs, 1):
            a, b = s.count("0"), s.count("1")
            for j in range(m + 1):
                for k in range(n + 1):
                    f[i][j][k] = f[i - 1][j][k]
                    if j >= a and k >= b:
                        f[i][j][k] = max(f[i][j][k], f[i - 1][j - a][k - b] + 1)
        return f[L][m][n]
```

**Độ phức tạp:** Time `O(L * m * n)`, Space `O(L * m * n)` (với `L = 600`, `m = n = 100`: khoảng `6 × 10^6` ô, vẫn chạy được nhưng tốn bộ nhớ).

### Cách 4 — DP 2 chiều nén bộ nhớ, duyệt NGƯỢC (cách nộp)

**Ý tưởng:** chỉ giữ `dp[j][k]` = "tốt nhất với tối đa `j` số 0, `k` số 1" sau khi xét một số chuỗi đầu. Khi xét chuỗi mới `(a, b)`:

```
for j từ m xuống a:
    for k từ n xuống b:
        dp[j][k] = max(dp[j][k], dp[j - a][k - b] + 1)
```

**Vì sao ngược?** `dp[j - a][k - b]` phải là giá trị **trước khi** xét chuỗi này (hàng `i - 1`). Nếu duyệt xuôi thì ô `(j - a, k - b)` (nhỏ hơn) đã được cập nhật bằng chính chuỗi này, ta sẽ **dùng chuỗi này hai lần** (xem Lỗi 1 ở mục 13). Duyệt từ lớn về nhỏ thì ô nhỏ chưa bị đụng tới.

Giới hạn vòng lặp `j >= a`, `k >= b` đã thay thế điều kiện "đủ chỗ", và "bỏ chuỗi" là mặc định (không đổi `dp`).

In [11]:
def find_max_form_2d(strs, m, n):
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for s in strs:
        a, b = s.count("0"), s.count("1")
        for j in range(m, a - 1, -1):
            for k in range(n, b - 1, -1):
                dp[j][k] = max(dp[j][k], dp[j - a][k - b] + 1)
    return dp[m][n]

for strs, m, n, ans in examples:
    print(find_max_form_2d(strs, m, n), "(mong doi", ans, ")")

4 (mong doi 4 )
2 (mong doi 2 )


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def findMaxForm(self, strs: List[str], m: int, n: int) -> int:
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for s in strs:
            a, b = s.count("0"), s.count("1")
            for j in range(m, a - 1, -1):
                for k in range(n, b - 1, -1):
                    dp[j][k] = max(dp[j][k], dp[j - a][k - b] + 1)
        return dp[m][n]
```

**Độ phức tạp:** Time `O(L * m * n)`, Space `O(m * n)`: chỉ `101 * 101 ≈ 10^4` ô.

## 8. TRACE từng bước

Ta theo dõi DP 2 chiều (Cách 4) trên **Ví dụ 2**: `strs = ["10","0","1"]`, `m = 1`, `n = 1`. Bảng `dp[j][k]` có kích thước `2 × 2`, ban đầu toàn `0`. Chuỗi đầu `"10"` có `(a, b) = (1, 1)`.

| Sau khi xét | `(a,b)` | ô cập nhật (theo thứ tự ngược) | `dp` (hàng `j`=0, `j`=1) |
|-------------|---------|-------------------------------|--------------------------|
| (ban đầu) | | | `[[0,0],[0,0]]` |
| `"10"` | (1,1) | `(1,1)`: `dp[0][0]+1 = 1` | `[[0,0],[0,1]]` |
| `"0"` | (1,0) | `(1,1)`: `max(1, dp[0][1]+1=1)`; `(1,0)`: `dp[0][0]+1 = 1` | `[[0,0],[1,1]]` |
| `"1"` | (0,1) | `(1,1)`: `max(1, dp[1][0]+1=2) = 2`; `(1,0)` không xét (k < b); `(0,1)`: `dp[0][0]+1 = 1` | `[[0,1],[1,2]]` |

Đáp án `dp[1][1] = 2`. Hàm dưới in đúng quá trình này.

In [12]:
def trace_dp2d(strs, m, n):
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    print("ban dau:", dp)
    for s in strs:
        a, b = s.count("0"), s.count("1")
        print(f"\nxet {s!r}: a = {a} so 0, b = {b} so 1")
        for j in range(m, a - 1, -1):
            for k in range(n, b - 1, -1):
                cu = dp[j][k]
                moi = max(cu, dp[j - a][k - b] + 1)
                note = "tang" if moi > cu else "giu"
                print(f"  dp[{j}][{k}]: max({cu}, dp[{j - a}][{k - b}] + 1 = {dp[j - a][k - b] + 1}) = {moi}  ({note})")
                dp[j][k] = moi
        print("  dp =", dp)
    print("\nDAP AN dp[m][n] =", dp[m][n])
    return dp[m][n]

trace_dp2d(["10", "0", "1"], 1, 1)

ban dau: [[0, 0], [0, 0]]

xet '10': a = 1 so 0, b = 1 so 1
  dp[1][1]: max(0, dp[0][0] + 1 = 1) = 1  (tang)
  dp = [[0, 0], [0, 1]]

xet '0': a = 1 so 0, b = 0 so 1
  dp[1][1]: max(1, dp[0][1] + 1 = 1) = 1  (giu)
  dp[1][0]: max(0, dp[0][0] + 1 = 1) = 1  (tang)
  dp = [[0, 0], [1, 1]]

xet '1': a = 0 so 0, b = 1 so 1
  dp[1][1]: max(1, dp[1][0] + 1 = 2) = 2  (tang)
  dp[0][1]: max(0, dp[0][0] + 1 = 1) = 1  (tang)
  dp = [[0, 1], [1, 2]]

DAP AN dp[m][n] = 2


2

Trace thứ hai: **Ví dụ 1** nhưng chỉ in hàng chéo `dp[j][k]` tại ô đích sau mỗi chuỗi để thấy câu trả lời tăng dần (`m = 5`, `n = 3`).

In [13]:
def trace_dich(strs, m, n):
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for s in strs:
        a, b = s.count("0"), s.count("1")
        for j in range(m, a - 1, -1):
            for k in range(n, b - 1, -1):
                dp[j][k] = max(dp[j][k], dp[j - a][k - b] + 1)
        print(f"sau {s!r:9} (0s={a}, 1s={b}): dp[{m}][{n}] = {dp[m][n]}")
    return dp[m][n]

trace_dich(["10", "0001", "111001", "1", "0"], 5, 3)

sau '10'      (0s=1, 1s=1): dp[5][3] = 1
sau '0001'    (0s=3, 1s=1): dp[5][3] = 2
sau '111001'  (0s=2, 1s=4): dp[5][3] = 2
sau '1'       (0s=0, 1s=1): dp[5][3] = 3
sau '0'       (0s=1, 1s=0): dp[5][3] = 4


4

## 9. Code nộp LeetCode

Ta nộp **Cách 4** (DP 2 chiều duyệt ngược): ngắn, nhanh nhất, tiết kiệm bộ nhớ.

```python
class Solution:
    def findMaxForm(self, strs: List[str], m: int, n: int) -> int:
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for s in strs:
            a, b = s.count("0"), s.count("1")
            for j in range(m, a - 1, -1):
                for k in range(n, b - 1, -1):
                    dp[j][k] = max(dp[j][k], dp[j - a][k - b] + 1)
        return dp[m][n]
```

In [14]:
class Solution:
    def findMaxForm(self, strs: List[str], m: int, n: int) -> int:
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for s in strs:
            a, b = s.count("0"), s.count("1")
            for j in range(m, a - 1, -1):
                for k in range(n, b - 1, -1):
                    dp[j][k] = max(dp[j][k], dp[j - a][k - b] + 1)
        return dp[m][n]

print(Solution().findMaxForm(["10", "0001", "111001", "1", "0"], 5, 3))   # 4
print(Solution().findMaxForm(["10", "0", "1"], 1, 1))                      # 2

4
2


## 10. Vì sao thuật toán đúng

**Định nghĩa bất biến.** Sau khi xử lý `i` chuỗi đầu, `dp[j][k]` bằng số chuỗi nhiều nhất chọn được từ `i` chuỗi đó với tổng `0` `<= j` và tổng `1` `<= k` (đặc tả "tối đa", nên `dp` **không giảm** khi `j` hay `k` tăng, và nền `0` hợp lệ cho mọi `(j, k)`).

**Bước quy nạp.** Với chuỗi mới `(a, b)` và túi `(j, k)`: một tập tối ưu hoặc **không** chứa chuỗi mới → giá trị `= dp_cũ[j][k]`, hoặc **có** chứa nó → phần còn lại là tập con của `i` chuỗi cũ dùng tối đa `(j - a, k - b)`, tối ưu là `dp_cũ[j - a][k - b]`, cộng `1`. Lấy `max` của hai khả năng đúng bằng giá trị tối ưu mới. Điều kiện `j >= a`, `k >= b` đảm bảo chuỗi mới vừa túi.

**Về thứ tự duyệt ngược.** Khi tính `dp[j][k]` ta cần `dp_cũ[j - a][k - b]`. Vì `a, b >= 0` và ít nhất một trong hai `> 0` (chuỗi không rỗng nên `a + b >= 1`), ô `(j - a, k - b)` luôn **nhỏ hơn** (theo thứ tự từ điển của `(j, k)` đang giảm dần) nên **chưa được cập nhật** ở vòng này. Vậy ta thực sự đọc giá trị cũ.

**Đáp án.** `dp[m][n]` sau khi xét hết `L` chuỗi.

## 11. Phân tích Time / Space Complexity

Gọi `L = len(strs)`, độ dài chuỗi `<= 100`.

| Cách | Time | Space | Ghi chú |
|------|------|-------|---------|
| 1. Vét cạn tập con | `O(2^L * L)` | `O(L)` | chỉ làm oracle (`L <= 14`) |
| 2. Đệ quy có nhớ | `O(L * m * n)` | `O(L * m * n)` | dễ nghĩ, hằng số lớn trong Python |
| 3. DP 3 chiều | `O(L * m * n)` | `O(L * m * n)` | dịch trực tiếp công thức |
| 4. DP 2 chiều (nộp) | `O(L * m * n + tổng độ dài)` | `O(m * n)` | tối ưu bộ nhớ |

Phần đếm `str.count` tốn thêm `O(tổng độ dài các chuỗi) <= 600 * 100 = 6 × 10^4`: nhỏ so với `6 × 10^6` của DP.

## 12. Bộ test

Ta dùng **vét cạn tập con** (Cách 1) làm oracle độc lập cho `L <= 12`, rồi đối chiếu Cách 2, 3, 4 và `Solution`; với input lớn đối chiếu 3 cách DP với nhau.

In [15]:
cac_cach = {
    "memo":     find_max_form_memo,
    "dp3d":     find_max_form_3d,
    "dp2d":     find_max_form_2d,
    "Solution": lambda strs, m, n: Solution().findMaxForm(list(strs), m, n),
}

def so_het(strs, m, n, oracle=None):
    if oracle is None:
        oracle = find_max_form_brute(strs, m, n)
    for ten, f in cac_cach.items():
        v = f(list(strs), m, n)
        assert v == oracle, (ten, strs, m, n, v, oracle)
    return oracle

# Ví dụ đề
for strs, m, n, ans in examples:
    assert so_het(strs, m, n, ans) == ans
print("Vi du de OK")

Vi du de OK


In [16]:
# Edge case
tests = [
    (["0"], 1, 1, 1),                                  # một chuỗi vừa túi
    (["0"], 1, 100, 1),
    (["1"], 100, 1, 1),
    (["00"], 1, 1, 0),                                 # không vừa số 0
    (["11"], 1, 1, 0),                                 # không vừa số 1
    (["0", "0", "0"], 2, 1, 2),                        # chỉ chọn 2 trong 3 chuỗi giống nhau
    (["01", "01", "01"], 2, 2, 2),
    (["10", "0", "1"], 1, 1, 2),
    (["0", "1"] * 5, 5, 5, 10),                        # tất cả đều vừa
    (["000", "111", "0011", "01"], 2, 2, 1),           # túi nhỏ: chỉ "01" vừa nhất
    (["10", "10", "10"], 3, 3, 3),                     # mỗi chuỗi (1,1): 3 chuỗi vừa (3,3)
    (["1" * 100], 100, 100, 1),                        # chuỗi dài 100
    (["0" * 100, "1" * 100], 100, 100, 2),             # hai chuỗi dài, mỗi cái dùng hết một loại
    (["111", "1000", "1000", "1000"], 9, 3, 3),        # ví dụ kinh điển khác
]
for strs, m, n, ans in tests:
    oracle = find_max_form_brute(strs, m, n) if len(strs) <= 12 else None
    kq = so_het(strs, m, n, oracle)
    assert kq == ans, (strs, m, n, kq, ans)
print("Edge case OK:", len(tests), "test")

Edge case OK: 14 test


**Phản ví dụ** cho heuristic "tham lam chọn chuỗi ngắn nhất/ít tốn nhất trước": ta để máy tìm trong mục 13 (Lỗi 3). Chỉ cần nhớ rằng bài knapsack 0-1 **không** giải bằng tham lam một cách tổng quát.

In [17]:
# Đối chiếu ngẫu nhiên có seed: L nhỏ với vét cạn
rng = random.Random(474)
for _ in range(500):
    L = rng.randint(1, 10)
    strs = ["".join(rng.choice("01") for _ in range(rng.randint(1, 6))) for _ in range(L)]
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    so_het(strs, m, n)
print("500 test ngau nhien (L <= 10) khop voi vet can")

500 test ngau nhien (L <= 10) khop voi vet can


In [18]:
# Đối chiếu 3 cách DP với nhau trên input cỡ vừa và lớn (ràng buộc đề)
rng = random.Random(4740)
for _ in range(15):
    L = rng.randint(30, 120)
    strs = ["".join(rng.choice("01") for _ in range(rng.randint(1, 12))) for _ in range(L)]
    m, n = rng.randint(1, 100), rng.randint(1, 100)
    a = find_max_form_3d(strs, m, n)
    b = find_max_form_2d(strs, m, n)
    c = find_max_form_memo(strs, m, n)
    d = Solution().findMaxForm(list(strs), m, n)
    assert a == b == c == d, (a, b, c, d)
print("15 test cỡ vừa: 3d == 2d == memo == Solution")

# Ca lớn nhất theo ràng buộc: 600 chuỗi, độ dài tới 100, m = n = 100
rng = random.Random(4741)
strs = ["".join(rng.choice("01") for _ in range(rng.randint(1, 100))) for _ in range(600)]
t0 = time.perf_counter()
kq_2d = Solution().findMaxForm(list(strs), 100, 100)
t_2d = time.perf_counter() - t0
t0 = time.perf_counter()
kq_3d = find_max_form_3d(strs, 100, 100)
t_3d = time.perf_counter() - t0
assert kq_2d == kq_3d
print(f"600 chuoi, m = n = 100: ket qua {kq_2d}; DP 2D {t_2d:.2f}s, DP 3D {t_3d:.2f}s")

# Ca lớn với chuỗi ngắn (nhiều chuỗi rẻ): đáp án lớn hơn
strs = ["".join(rng.choice("01") for _ in range(rng.randint(1, 3))) for _ in range(600)]
kq = Solution().findMaxForm(list(strs), 100, 100)
assert kq == find_max_form_3d(strs, 100, 100)
print("600 chuoi ngan, m = n = 100: ket qua", kq)

15 test cỡ vừa: 3d == 2d == memo == Solution


600 chuoi, m = n = 100: ket qua 50; DP 2D 0.26s, DP 3D 0.94s


600 chuoi ngan, m = n = 100: ket qua 199


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Duyệt XUÔI trong DP 2 chiều (dùng một chuỗi nhiều lần)

Nếu viết `for j in range(a, m + 1)` và `for k in range(b, n + 1)` (xuôi), thì khi tính `dp[j][k]` ta đọc `dp[j - a][k - b]` đã bị **chính chuỗi này** cập nhật, nghĩa là ta lấy chuỗi đó nhiều lần: biến bài thành knapsack **vô hạn** (unbounded).

In [19]:
def sai_duyet_xuoi(strs, m, n):
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for s in strs:
        a, b = s.count("0"), s.count("1")
        for j in range(a, m + 1):                 # SAI: duyệt xuôi
            for k in range(b, n + 1):
                dp[j][k] = max(dp[j][k], dp[j - a][k - b] + 1)
    return dp[m][n]

print("sai :", sai_duyet_xuoi(["0"], 3, 3))        # 3: lấy chuỗi "0" ba lần!
print("dung:", Solution().findMaxForm(["0"], 3, 3)) # 1: chỉ có MỘT chuỗi "0"
print("sai :", sai_duyet_xuoi(["10", "0", "1"], 3, 3), "| dung:", Solution().findMaxForm(["10", "0", "1"], 3, 3))

sai : 3
dung: 1
sai : 6 | dung: 3


**Vì sao sai:** với `["0"]` và túi `(3, 3)`: `dp[1][0]` được `1`, rồi `dp[2][0]` đọc `dp[1][0] + 1 = 2` (vừa cập nhật ở cùng vòng chuỗi), `dp[3][0] = 3`. Một chuỗi bị đếm ba lần.

**Code đúng:** duyệt `j` từ `m` xuống `a` và `k` từ `n` xuống `b` (mục 7, Cách 4).

### Lỗi 2 — Nhầm `m` là số `1` và `n` là số `0`

Đề nói **`m` số `0`** và **`n` số `1`**. Nếu đổi chỗ (`a` là số `1`, `b` là số `0`) kết quả có thể sai khi `m != n`.

In [20]:
def sai_nham_m_n(strs, m, n):
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for s in strs:
        a, b = s.count("1"), s.count("0")         # SAI: a phải là số 0, b là số 1
        for j in range(m, a - 1, -1):
            for k in range(n, b - 1, -1):
                dp[j][k] = max(dp[j][k], dp[j - a][k - b] + 1)
    return dp[m][n]

strs, m, n = ["10", "0001", "111001", "1", "0"], 5, 3
print("sai :", sai_nham_m_n(strs, m, n), "| dung:", Solution().findMaxForm(strs, m, n))
strs, m, n = ["000", "000"], 6, 1
print("sai :", sai_nham_m_n(strs, m, n), "| dung:", Solution().findMaxForm(strs, m, n))

sai : 3 | dung: 4
sai : 0 | dung: 2


**Vì sao sai:** chiều `j` của bảng ứng với "tối đa `m` số 0" nên `a` phải là `s.count("0")`. Nhầm hai số đếm làm chuỗi `"000"` bị tính nặng ở chiều sai. Cách phòng: đặt tên rõ `zeros`, `ones`.

**Code đúng:** `a, b = s.count("0"), s.count("1")` và `dp[j][k]` với `j` ↔ số 0, `k` ↔ số 1.

### Lỗi 3 — Tham lam: chọn chuỗi "rẻ" nhất trước

Ý tưởng hấp dẫn: sắp chuỗi theo độ dài (hoặc `a + b`) tăng dần rồi lấy khi còn chỗ. Ta để máy tìm phản ví dụ.

In [21]:
def tham_lam(strs, m, n):
    dem = 0
    for s in sorted(strs, key=len):               # ngắn nhất trước
        a, b = s.count("0"), s.count("1")
        if m >= a and n >= b:
            m -= a
            n -= b
            dem += 1
    return dem

rng = random.Random(7)
tim = None
for _ in range(100000):
    L = rng.randint(2, 6)
    strs = ["".join(rng.choice("01") for _ in range(rng.randint(1, 4))) for _ in range(L)]
    m, n = rng.randint(1, 4), rng.randint(1, 4)
    if tham_lam(strs, m, n) != Solution().findMaxForm(list(strs), m, n):
        tim = (strs, m, n)
        break
strs, m, n = tim
print("phan vi du:", strs, m, n, "| tham lam =", tham_lam(strs, m, n), "| dung =", Solution().findMaxForm(list(strs), m, n))

phan vi du: ['10', '0', '11', '111', '000', '1'] 4 3 | tham lam = 3 | dung = 4


**Vì sao sai:** "độ dài ngắn" không phản ánh việc chuỗi tốn loại tài nguyên nào. Hai chuỗi ngắn cùng nặng về số `0` có thể chặn nhau, trong khi chọn chuỗi dài hơn nhưng nhẹ ở loại tài nguyên khan hiếm lại cho nhiều chuỗi hơn. Đây là knapsack nên cần DP.

### Lỗi 4 — Quên trường hợp "bỏ chuỗi" trong DP 3 chiều

Nếu quên dòng `f[i][j][k] = f[i-1][j][k]`, các ô mà chuỗi `i` không vừa sẽ giữ `0` thay vì kế thừa kết quả cũ.

In [22]:
def sai_quen_bo(strs, m, n):
    L = len(strs)
    f = [[[0] * (n + 1) for _ in range(m + 1)] for _ in range(L + 1)]
    for i, s in enumerate(strs, 1):
        a, b = s.count("0"), s.count("1")
        for j in range(m + 1):
            for k in range(n + 1):
                # SAI: thiếu f[i][j][k] = f[i-1][j][k]
                if j >= a and k >= b:
                    f[i][j][k] = f[i - 1][j - a][k - b] + 1
    return f[L][m][n]

print("sai :", sai_quen_bo(["10", "0001", "111001", "1", "0"], 5, 3), "| dung: 4")
print("sai :", sai_quen_bo(["1", "11"], 1, 1), "| dung:", Solution().findMaxForm(["1", "11"], 1, 1))

sai : 2 | dung: 4
sai : 0 | dung: 1


**Vì sao sai:** nếu chuỗi cuối (ví dụ `"11"`) không vừa túi `(1, 1)` thì `f[L][1][1]` phải bằng `f[L-1][1][1]`, nhưng bản sai trả `0`.

### Lỗi 5 — Tạo bảng bằng `[[0] * n] * m`

Như mục 3.3: các hàng dùng chung một list nên mọi cập nhật "lan" sang hàng khác. Kết quả sai rất khó đoán.

In [23]:
def sai_bang_chung_hang(strs, m, n):
    dp = [[0] * (n + 1)] * (m + 1)                # SAI: các hàng là cùng một đối tượng
    for s in strs:
        a, b = s.count("0"), s.count("1")
        for j in range(m, a - 1, -1):
            for k in range(n, b - 1, -1):
                dp[j][k] = max(dp[j][k], dp[j - a][k - b] + 1)
    return dp[m][n]

print("sai :", sai_bang_chung_hang(["10", "0", "1"], 1, 1), "| dung:", Solution().findMaxForm(["10", "0", "1"], 1, 1))
print("sai :", sai_bang_chung_hang(["10", "0001", "111001", "1", "0"], 5, 3), "| dung: 4")

sai : 2 | dung: 2
sai : 8 | dung: 4


**Code đúng:** `dp = [[0] * (n + 1) for _ in range(m + 1)]`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu knapsack 0-1:** "chọn một tập con, mỗi phần tử tối đa một lần, tổng tài nguyên bị chặn, tối đa hóa số lượng/giá trị".
- **Mỗi tài nguyên = một chiều của bảng.** Một loại tài nguyên → `dp[c]`; hai loại (số 0, số 1) → `dp[j][k]`; ba loại → `dp[i][j][k]`.
- **Quy tắc duyệt:** 0-1 knapsack nén bộ nhớ → duyệt **ngược** (từ sức chứa lớn xuống nhỏ); knapsack vô hạn → duyệt **xuôi**.
- **Giá trị bằng 1:** khi chỉ đếm số phần tử, giá trị mỗi món là `1` (ở đây `+ 1`).
- **Kiểm tra nhanh tính đúng:** `dp` phải không giảm theo `j`, `k`; `dp[m][n] <= L`.
- **Bài họ hàng:** LeetCode 416 *Partition Equal Subset Sum*, 494 *Target Sum*, 1049 *Last Stone Weight II*, 879 *Profitable Schemes*, 2518 *Number of Great Partitions*.

## 15. Biến thể

1. **Mỗi chuỗi dùng được vô hạn lần** (unbounded knapsack): chính là "lỗi duyệt xuôi" nhưng làm có chủ đích: chạy `j`, `k` xuôi.
2. **Mỗi chuỗi có giá trị riêng `v`**: thay `+ 1` bằng `+ v`.
3. **Yêu cầu dùng đúng `m` số 0 và `n` số 1** (không phải "tối đa"): khởi tạo `dp` với `-inf` ngoại trừ `dp[0][0] = 0`.
4. **Ba loại tài nguyên** (ví dụ `'0'`, `'1'`, `'2'`): thêm một chiều thứ ba cho bảng.
5. **Trả về chính tập con** (không chỉ kích thước): dùng bảng 3 chiều rồi truy vết ngược từ `f[L][m][n]`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Vét cạn tập con | thử `2^L` tập, kiểm tra tài nguyên | `O(2^L * L)` | `O(L)` | oracle cho `L <= 14`, hiểu đề |
| 2. Đệ quy có nhớ | `go(i, j, k)` lấy/bỏ, nhớ kết quả | `O(L m n)` | `O(L m n)` | khi dễ viết nhất, ít trạng thái đạt được |
| 3. DP 3 chiều | `f[i][j][k]` từ hàng `i-1` | `O(L m n)` | `O(L m n)` | cần truy vết tập chọn |
| 4. DP 2 chiều duyệt ngược | nén hàng, giữ `dp[j][k]` | `O(L m n)` | `O(m n)` | **nộp bài**, nhanh và gọn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `"10010".count("0")`; (b) `list(range(4, 0, -1))`; (c) `Solution().findMaxForm(["0"], 1, 1)`; (d) `Solution().findMaxForm(["00","11"], 1, 1)`; (e) `Solution().findMaxForm(["0","0","0"], 2, 5)`.

**Bài 2 (tính tay).** Với `strs = ["10","0","1"]`, `m = 2`, `n = 1`: liệt kê giá `(số 0, số 1)` từng chuỗi, nêu tập con lớn nhất và kích thước.

**Bài 3 (điền TODO).** Hoàn thành `cap_gia(s)` trả về tuple `(số 0, số 1)` của chuỗi `s`, và `them_mot_chuoi(dp, s, m, n)` cập nhật bảng `dp` tại chỗ cho một chuỗi (duyệt ngược).

In [24]:
def cap_gia(s):
    # TODO: trả về (s.count("0"), s.count("1"))
    return None

def them_mot_chuoi(dp, s, m, n):
    # TODO 1: a, b = cap_gia(s)
    # TODO 2: for j từ m xuống a, for k từ n xuống b: dp[j][k] = max(dp[j][k], dp[j-a][k-b] + 1)
    pass

# KHÔNG gọi cap_gia / them_mot_chuoi ở đây để "Run All" không lỗi (chưa điền xong).

**Bài 4 (tự viết lại).** Viết `find_max_form_todo(strs, m, n)` bằng DP 2 chiều **không nhìn lại** mục 7. Kiểm tra bằng hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** Viết `find_max_form_gia_tri(items, m, n)` với `items` là danh sách `(chuỗi, giá_trị)`: mục tiêu là **tổng giá trị lớn nhất** (không phải số chuỗi) với ràng buộc cũ. Kiểm tra: `find_max_form_gia_tri([("0",5),("1",4),("01",10)], 1, 1) == 10` (lấy `"01"` vì `10 > 5 + 4 = 9`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: (a) `3`; (b) `[4, 3, 2, 1]`; (c) `1`; (d) `0` (mỗi chuỗi cần hai ký tự cùng loại, vượt `1`); (e) `2` (chỉ còn `2` số `0`). Bài 3: `cap_gia` chỉ một dòng.
- **Mức 2 (vừa):** Bài 2: giá `(1,1)`, `(1,0)`, `(0,1)`; túi `(2,1)`: `{"0","1"}` dùng `(1,1)`; thêm `"10"` thành `(2,2)` vượt `n`; thử `{"10","0"}`: `(2,1)` ✓ cũng 2 chuỗi; đáp án 2. Bài 4: nhớ duyệt ngược.
- **Mức 3 (sâu):** Bài 5: thay `+ 1` bằng `+ giá_trị`; với `"01"` giá `(1,1)` và giá trị `10` thì `dp[1][1] = max(5, 4, 10)`: chú ý vì hai chuỗi `"0"` và `"1"` cùng lấy được thì `5 + 4 = 9 < 10`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
assert "10010".count("0") == 3
assert list(range(4, 0, -1)) == [4, 3, 2, 1]
assert Solution().findMaxForm(["0"], 1, 1) == 1
assert Solution().findMaxForm(["00", "11"], 1, 1) == 0
assert Solution().findMaxForm(["0", "0", "0"], 2, 5) == 2
print("Bài 1 OK")

# Bài 2
strs, m, n = ["10", "0", "1"], 2, 1
print("Bài 2: gia =", [(s.count("0"), s.count("1")) for s in strs], "dap an =", Solution().findMaxForm(strs, m, n))
assert Solution().findMaxForm(strs, m, n) == 2 == find_max_form_brute(strs, m, n)
print("Bài 2 OK")

# Bài 3
def cap_gia_ans(s):
    return (s.count("0"), s.count("1"))

def them_mot_chuoi_ans(dp, s, m, n):
    a, b = cap_gia_ans(s)
    for j in range(m, a - 1, -1):
        for k in range(n, b - 1, -1):
            dp[j][k] = max(dp[j][k], dp[j - a][k - b] + 1)

assert cap_gia_ans("111001") == (2, 4)
dp = [[0] * 4 for _ in range(6)]
for s in ["10", "0001", "111001", "1", "0"]:
    them_mot_chuoi_ans(dp, s, 5, 3)
assert dp[5][3] == 4
print("Bài 3 OK")

# Bài 4
def find_max_form_todo_ans(strs, m, n):
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for s in strs:
        a, b = s.count("0"), s.count("1")
        for j in range(m, a - 1, -1):
            for k in range(n, b - 1, -1):
                dp[j][k] = max(dp[j][k], dp[j - a][k - b] + 1)
    return dp[m][n]

for strs, m, n, ans in examples:
    assert find_max_form_todo_ans(strs, m, n) == ans
random.seed(44)
for _ in range(200):
    strs = ["".join(random.choice("01") for _ in range(random.randint(1, 5))) for _ in range(random.randint(1, 9))]
    m, n = random.randint(1, 6), random.randint(1, 6)
    assert find_max_form_todo_ans(strs, m, n) == find_max_form_brute(strs, m, n)
print("Bài 4 OK")

# Bài 5
def find_max_form_gia_tri(items, m, n):
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for s, v in items:
        a, b = s.count("0"), s.count("1")
        for j in range(m, a - 1, -1):
            for k in range(n, b - 1, -1):
                dp[j][k] = max(dp[j][k], dp[j - a][k - b] + v)
    return dp[m][n]

assert find_max_form_gia_tri([("0", 5), ("1", 4), ("01", 10)], 1, 1) == 10
assert find_max_form_gia_tri([("0", 5), ("1", 4), ("01", 8)], 1, 1) == 9
assert find_max_form_gia_tri([("0", 1), ("1", 1)], 1, 1) == Solution().findMaxForm(["0", "1"], 1, 1)
print("Bài 5 OK")

Bài 1 OK
Bài 2: gia = [(1, 1), (1, 0), (0, 1)] dap an = 2
Bài 2 OK
Bài 3 OK
Bài 4 OK
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (knapsack 0-1 hai chiều, nén bộ nhớ):**

```python
dp = [[0] * (n + 1) for _ in range(m + 1)]
for s in strs:
    a, b = s.count("0"), s.count("1")           # "giá" của món: (số 0, số 1)
    for j in range(m, a - 1, -1):               # NGƯỢC
        for k in range(n, b - 1, -1):           # NGƯỢC
            dp[j][k] = max(dp[j][k], dp[j - a][k - b] + 1)
return dp[m][n]
```

**Checklist:**

- [ ] `a` là số `0` (ứng `m`), `b` là số `1` (ứng `n`): không đảo.
- [ ] Mỗi chuỗi chỉ được chọn **một lần** → vòng `j`, `k` duyệt **từ lớn về nhỏ**.
- [ ] Bảng tạo bằng list comprehension, kích thước `(m + 1) × (n + 1)`, khởi tạo `0`.
- [ ] Điều kiện vừa túi nằm trong cận vòng lặp `range(m, a - 1, -1)`, `range(n, b - 1, -1)`.
- [ ] Không dùng tham lam; đối chiếu với vét cạn trên input nhỏ.
- [ ] Thử tay `["10","0","1"], 1, 1` (ra `2`) và `["0"], 3, 3` (ra `1`, không phải `3`).
- [ ] Đúng signature `class Solution` / `findMaxForm(self, strs: List[str], m: int, n: int) -> int`; độ phức tạp `O(L * m * n)` thời gian, `O(m * n)` bộ nhớ.